In [5]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.cloud import storage
import vertexai
import numpy as np

load_dotenv() 

PROJECT_ID = os.environ["PROJECT_ID"]
LOCATION = os.environ["LOCATION"]

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"./service_account.json"

In [6]:
aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket="gs://ridwan-faturrahman-bucket-123456"
)

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION
)

### Check Custom Job

In [2]:
jobs = aiplatform.CustomJob.list()

In [3]:
print(len(jobs))
for job in jobs:
    print(job.display_name, job.resource_name)

0


### Check Artifact Registry

In [10]:
from google.cloud import artifactregistry_v1

In [14]:
def list_repositories():
    client = artifactregistry_v1.ArtifactRegistryClient()
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"
    request = artifactregistry_v1.ListRepositoriesRequest(parent=parent)
    page_result = client.list_repositories(request=request)

    for repo in page_result:
        print(f"- Name: {repo.name.split('/')[-1]}")
        print(f"  Format: {repo.format_.name}")
        print(f"  Description: {repo.description}\n")

In [15]:
list_repositories()

- Name: api-cvinsight-repo
  Format: DOCKER
  Description: API CV Insight Docker images

- Name: cvinsight-repo
  Format: DOCKER
  Description: CV Insight Docker images

- Name: fast-api-alembic-repo
  Format: DOCKER
  Description: Fast API Alembic Docker images

- Name: fastapi-alembic-repo
  Format: DOCKER
  Description: Fast API Alembic Docker images

- Name: random-forest-classifier-repo
  Format: DOCKER
  Description: 

- Name: train-tensorflow-repo
  Format: DOCKER
  Description: Train Tensorflow Model



In [16]:
def list_docker_images():
    client = artifactregistry_v1.ArtifactRegistryClient()

    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/repositories/{REPO}"

    request = artifactregistry_v1.ListDockerImagesRequest(parent=parent)
    images = client.list_docker_images(request=request)

    found = False
    for image in images:
        found = True
        print("Image URI:", image.uri)
        print("Tags:", image.tags)
        print("Update time:", image.update_time)
        print()

    if not found:
        print("Repository kosong (belum ada Docker image).")

In [17]:
REPO = "train-tensorflow-repo"

In [18]:
list_docker_images()

Image URI: us-central1-docker.pkg.dev/ridwan-faturrahman/train-tensorflow-repo/train-tensorflow-app@sha256:b60ab814dadb027a4c2e977763174c182e59369e110a1e1501e14a0b61399dfe
Tags: []
Update time: 2026-03-05 07:16:56.258287+00:00

Image URI: us-central1-docker.pkg.dev/ridwan-faturrahman/train-tensorflow-repo/train-tensorflow-app@sha256:c67c02b0a5c83f6740d7cd249e56d2e50b52be5ee8f29c0292da87568b31c630
Tags: []
Update time: 2026-03-05 07:16:54.966863+00:00

Image URI: us-central1-docker.pkg.dev/ridwan-faturrahman/train-tensorflow-repo/train-tensorflow-app@sha256:d540be6365d749b617dcd4243e9ea33a6affbc88e400010df2b09726d47a5593
Tags: ['latest']
Update time: 2026-03-05 07:16:56.258287+00:00



### Upload Job

In [7]:
job = aiplatform.CustomContainerTrainingJob(
    display_name="train-mpg-model",
    container_uri="us-central1-docker.pkg.dev/ridwan-faturrahman/train-tensorflow-repo/train-tensorflow-app:latest",
)

In [19]:
model = job.run(
    replica_count=1,
    machine_type="e2-standard-4",
    args=[
        "--bucket", "gs://ridwan-faturrahman-bucket-123456",
        "--epochs", "1"
    ]
)

Training Output directory:
gs://ridwan-faturrahman-bucket-123456/aiplatform-custom-training-2026-03-05-14:49:17.345 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/2242739751973027840?project=344969539300
CustomContainerTrainingJob projects/344969539300/locations/us-central1/trainingPipelines/2242739751973027840 current state:
2
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/555873410685599744?project=344969539300
CustomContainerTrainingJob projects/344969539300/locations/us-central1/trainingPipelines/2242739751973027840 current state:
3
CustomContainerTrainingJob projects/344969539300/locations/us-central1/trainingPipelines/2242739751973027840 current state:
3
CustomContainerTrainingJob projects/344969539300/locations/us-central1/trainingPipelines/2242739751973027840 current state:
3
CustomContainerTrainingJob projects/344969539300/locations/us-central1/trainingPipelines/22427397519730278